# MIT103 Assessment 3 | Guided Starter Notebook
**Personal Finance Tracker, Phase 2**

This is a scaffold, not a completed solution. Replace every `TODO` with your own implementation. You may reuse and improve your Assessment 2 code. Follow the assessment PDF for all requirements.

**Colab storage warning:** files saved only in `/content` can disappear when the runtime resets. Demonstrate a genuine restore from Google Drive or an uploaded CSV. Never upload real credentials or financial records.

## Step 0. Load supplied sample data
Upload the supplied `MIT103_A3_Revised_T3_2026_Sample_Transactions.csv` in Colab using the Files panel, or mount Drive and specify its path. Keep the original supplied CSV as an untouched backup. The app should write to a **separate working CSV**. The example rows are synthetic.

In [2]:

from pathlib import Path
import csv
import hashlib
import hmac
import os
import shutil
from google.colab import files

# Upload the original sample CSV file
uploaded = files.upload()

# Set file paths
SAMPLE_CSV = Path("MIT103_A3_Revised_T3_2026_Sample_Transactions.csv")
WORKING_CSV = Path("my_transactions.csv")

# Create a separate working copy
if SAMPLE_CSV.exists():
    if not WORKING_CSV.exists():
        shutil.copy2(SAMPLE_CSV, WORKING_CSV)
        print("Working CSV created successfully.")
    else:
        print("Working CSV already exists.")

    print("Sample file available:", SAMPLE_CSV.exists())
    print("Working file available:", WORKING_CSV.exists())
else:
    print("Error: Sample CSV file not found.")

Saving MIT103_A3_Revised_T3_2026_Sample_Transactions.csv to MIT103_A3_Revised_T3_2026_Sample_Transactions.csv
Working CSV created successfully.
Sample file available: True
Working file available: True


## Step 1. Transaction model and core program
**Foundation:** complete the class, menu, add, display, search and delete operations. You can replace these blocks with your Assessment 2 solution if you explain changes. Store transactions in a list of `Transaction` objects.

In [3]:

# Transaction class
class Transaction:
    def __init__(self, transaction_id, date, category, description, amount):
        self.transaction_id = transaction_id
        self.date = date
        self.category = category
        self.description = description
        self.amount = float(amount)

    def __str__(self):
        return (
            f"ID: {self.transaction_id} | "
            f"Date: {self.date} | "
            f"Category: {self.category} | "
            f"Description: {self.description} | "
            f"Amount: {self.amount:.2f}"
        )


# Add a new transaction
def add_transaction(transactions):
    print("\n--- Add New Transaction ---")

    transaction_id = input("Enter Transaction ID: ").strip()

    if not transaction_id:
        print("Transaction ID cannot be empty.")
        return

    for transaction in transactions:
        if transaction.transaction_id.lower() == transaction_id.lower():
            print("This Transaction ID already exists.")
            return

    date = input("Enter Date (DD-MM-YYYY): ").strip()
    category = input("Enter Category: ").strip()
    description = input("Enter Description: ").strip()

    if not date or not category or not description:
        print("All fields are required.")
        return

    try:
        amount = float(input("Enter Amount: "))

        if amount == 0:
            print("Amount cannot be zero.")
            return

    except ValueError:
        print("Please enter a valid numeric amount.")
        return

    new_transaction = Transaction(
        transaction_id, date, category, description, amount
    )

    transactions.append(new_transaction)
    print("Transaction added successfully.")


# Display all transactions
def display_transactions(transactions):
    print("\n--- All Transactions ---")

    if not transactions:
        print("No transactions available.")
        return

    for transaction in transactions:
        print(transaction)


# Search transactions by ID or category
def search_transactions(transactions):
    print("\n--- Search Transactions ---")

    search_value = input("Enter Transaction ID or Category: ").strip()

    if not search_value:
        print("Please enter a search value.")
        return

    found = False

    for transaction in transactions:
        if (transaction.transaction_id.lower() == search_value.lower()
                or transaction.category.lower() == search_value.lower()):

            print(transaction)
            found = True

    if not found:
        print("No matching transaction found.")


# Delete a transaction
def delete_transaction(transactions, transaction_id, role):

    if role.lower() != "admin":
        print("Access denied. Only Admin can delete transactions.")
        return False

    for transaction in transactions:
        if transaction.transaction_id.lower() == transaction_id.lower():
            transactions.remove(transaction)
            print("Transaction deleted successfully.")
            return True

    print("Transaction ID not found.")
    return False


## Step 2. CSV persistence
Use the supplied CSV header exactly. Read records into `Transaction` objects. Save updates to a working file, not the original supplied file. Explain your storage location and demonstrate restoration after a fresh execution/session.

In [4]:

CSV_FIELDS = [
    "transaction_id",
    "date",
    "category",
    "description",
    "amount"
]


# Load transactions from CSV
def load_transactions(filename):
    transactions = []

    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)

            if reader.fieldnames != CSV_FIELDS:
                print("Error: CSV headers are incorrect.")
                return transactions

            for row in reader:
                try:
                    transaction = Transaction(
                        row["transaction_id"],
                        row["date"],
                        row["category"],
                        row["description"],
                        float(row["amount"])
                    )

                    transactions.append(transaction)

                except (ValueError, TypeError, KeyError):
                    print("Invalid row skipped.")

        print("Transactions loaded successfully.")

    except FileNotFoundError:
        print("File not found. Starting with an empty list.")

    except OSError:
        print("Error while reading the CSV file.")

    return transactions


# Save transactions to CSV
def save_transactions(transactions, filename):

    file_path = Path(filename)

    if file_path.resolve() == SAMPLE_CSV.resolve():
        print("Error: Original sample file cannot be overwritten.")
        return False

    try:
        with open(file_path, "w", newline="", encoding="utf-8") as file:

            writer = csv.DictWriter(file, fieldnames=CSV_FIELDS)
            writer.writeheader()

            for transaction in transactions:
                writer.writerow({
                    "transaction_id": transaction.transaction_id,
                    "date": transaction.date,
                    "category": transaction.category,
                    "description": transaction.description,
                    "amount": transaction.amount
                })

        print("Transactions saved successfully.")
        return True

    except OSError:
        print("Error while saving the CSV file.")
        return False


## Step 3. Authentication and permissions
Use two **fictional** accounts, `admin` and `viewer`. The Viewer may display/search, but not add/delete. Store only salted password verifiers, never plaintext passwords in the repository. For a classroom demo, generate demo credentials locally and explain setup in the report. Do not commit credential-generation output.

In [5]:

# Load transactions from the working CSV file
transactions = load_transactions(WORKING_CSV)

# Display all loaded transactions
display_transactions(transactions)

# Show the total number of transactions
print("\nTotal transactions:", len(transactions))


Transactions loaded successfully.

--- All Transactions ---
ID: T001 | Date: 02/11/2026 | Category: Food | Description: Groceries | Amount: -48.50
ID: T002 | Date: 03/11/2026 | Category: Transport | Description: Train card | Amount: -25.00
ID: T003 | Date: 04/11/2026 | Category: Income | Description: Casual shift | Amount: 180.00
ID: T004 | Date: 05/11/2026 | Category: Food | Description: Lunch | Amount: -14.20
ID: T005 | Date: 06/11/2026 | Category: Bills | Description: Mobile plan | Amount: -35.00
ID: T006 | Date: 07/11/2026 | Category: Transport | Description: Bus ticket | Amount: -5.60
ID: T007 | Date: 08/11/2026 | Category: Income | Description: Refund | Amount: 20.00
ID: T008 | Date: 09/11/2026 | Category: Food | Description: Coffee | Amount: -6.50

Total transactions: 8


In [6]:

import secrets
import getpass

# Create a secure password verifier
def make_password_verifier(password):
    salt = secrets.token_bytes(16)

    digest = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt,
        100000
    )

    return salt.hex() + ":" + digest.hex()


# Verify the entered password
def verify_password(password, stored_verifier):
    try:
        salt_hex, stored_hash = stored_verifier.split(":")
        salt = bytes.fromhex(salt_hex)

        new_hash = hashlib.pbkdf2_hmac(
            "sha256",
            password.encode("utf-8"),
            salt,
            100000
        )

        return hmac.compare_digest(
            new_hash.hex(),
            stored_hash
        )

    except (ValueError, TypeError):
        return False


# Login function
def login(accounts):
    username = input("Enter username: ").strip().lower()
    password = getpass.getpass("Enter password: ")

    if username in accounts:
        if verify_password(password, accounts[username]["password"]):
            print("Login successful.")
            return accounts[username]["role"]

    print("Invalid username or password.")
    return None


# Check user permissions
def is_authorised(role, action):

    permissions = {
        "admin": ["display", "search", "add", "delete"],
        "viewer": ["display", "search"]
    }

    return action in permissions.get(role, [])


# Create two fictional demonstration accounts
admin_password = getpass.getpass("Create Admin password: ")
viewer_password = getpass.getpass("Create Viewer password: ")

accounts = {
    "admin": {
        "role": "admin",
        "password": make_password_verifier(admin_password)
    },
    "viewer": {
        "role": "viewer",
        "password": make_password_verifier(viewer_password)
    }
}

print("Admin and Viewer accounts created successfully.")


Create Admin password: ··········
Create Viewer password: ··········
Admin and Viewer accounts created successfully.


## Step 4. Improved data organisation
Use a dictionary for a meaningful purpose (for example, totals grouped by category or ID lookup). Explain what it improves and what it does not improve.

In [7]:

def category_totals(transactions):
    totals = {}

    for transaction in transactions:
        category = transaction.category

        if category in totals:
            totals[category] += transaction.amount
        else:
            totals[category] = transaction.amount

    return totals


In [8]:

print("Category-wise totals:")
for category, total in category_totals(transactions).items():
    print(category, ":", round(total, 2))


Category-wise totals:
Food : -69.2
Transport : -30.6
Income : 200.0
Bills : -35.0


# Explanation:
I used the dictionary function to get totals of each category. This is useful so that we can see how much money has been spent/incoming in certain categories.

The data is well structured but errors in the transactions have not been rectified and duplication has not been avoided.



## Step 5. Connect the menu
Use `while`, `if/elif/else`, function calls and error handling. Apply role checks inside protected operations, not only in the visible menu. Save after a successful add/delete.

In [9]:

# Add a transaction with role permission and input validation
def add_transaction(transactions, role):

    if not is_authorised(role, "add"):
        print("Access denied. Only Admin can add transactions.")
        return False

    transaction_id = input("Enter transaction ID: ").strip()

    if not transaction_id:
        print("Transaction ID cannot be empty.")
        return False

    for transaction in transactions:
        if transaction.transaction_id.lower() == transaction_id.lower():
            print("This transaction ID already exists.")
            return False

    date = input("Enter date (DD/MM/YYYY): ").strip()

    try:
        from datetime import datetime
        datetime.strptime(date, "%d/%m/%Y")
    except ValueError:
        print("Invalid date. Please use DD/MM/YYYY.")
        return False

    category = input("Enter category: ").strip()
    description = input("Enter description: ").strip()

    if not category or not description:
        print("Category and description cannot be empty.")
        return False

    try:
        amount = float(input("Enter amount: "))
    except ValueError:
        print("Invalid amount. Please enter a number.")
        return False

    new_transaction = Transaction(
        transaction_id, date, category, description, amount
    )

    transactions.append(new_transaction)
    print("Transaction added successfully.")
    return True


# Main application menu
def main():

    # Load transactions from the working CSV
    if not WORKING_CSV.exists():
        if SAMPLE_CSV.exists():
            initial_transactions = load_transactions(SAMPLE_CSV)
            save_transactions(initial_transactions, WORKING_CSV)
        else:
            print("Sample CSV file not found.")
            return

    transactions = load_transactions(WORKING_CSV)

    print("\n===== Personal Finance Tracker =====")

    # Login
    role = login(accounts)

    if role is None:
        print("Login failed. Exiting program.")
        return

    print("\nWelcome! Your role is:", role.upper())

    # Repeated menu
    while True:

        print("\n===== MAIN MENU =====")

        if role == "admin":
            print("1. Add Transaction")
            print("2. Display Transactions")
            print("3. Search Transaction")
            print("4. Delete Transaction")
            print("5. Exit")

        else:
            print("1. Display Transactions")
            print("2. Search Transaction")
            print("3. Exit")

        choice = input("Enter your choice: ").strip()

        if role == "admin":

            if choice == "1":
                if add_transaction(transactions, role):
                    save_transactions(transactions, WORKING_CSV)

            elif choice == "2":
                display_transactions(transactions)

            elif choice == "3":
                search_transactions(transactions)

            elif choice == "4":
                transaction_id = input(
                    "Enter transaction ID to delete: "
                ).strip()

                if delete_transaction(
                    transactions, transaction_id, role
                ):
                    save_transactions(transactions, WORKING_CSV)

            elif choice == "5":
                print("Thank you for using the Finance Tracker.")
                break

            else:
                print("Invalid menu choice. Please try again.")

        else:

            if choice == "1":
                display_transactions(transactions)

            elif choice == "2":
                search_transactions(transactions)

            elif choice == "3":
                print("Thank you for using the Finance Tracker.")
                break

            else:
                print("Invalid menu choice. Please try again.")


# Start the application
main()


Transactions loaded successfully.

===== Personal Finance Tracker =====
Enter username: Admin
Enter password: ··········
Login successful.

Welcome! Your role is: ADMIN

===== MAIN MENU =====
1. Add Transaction
2. Display Transactions
3. Search Transaction
4. Delete Transaction
5. Exit
Enter your choice: 1
Enter transaction ID: T009
Enter date (DD/MM/YYYY): 10/11/2026
Enter category: Food
Enter description: Grocery shopping
Enter amount: -25.50
Transaction added successfully.
Transactions saved successfully.

===== MAIN MENU =====
1. Add Transaction
2. Display Transactions
3. Search Transaction
4. Delete Transaction
5. Exit
Enter your choice: 2

--- All Transactions ---
ID: T001 | Date: 02/11/2026 | Category: Food | Description: Groceries | Amount: -48.50
ID: T002 | Date: 03/11/2026 | Category: Transport | Description: Train card | Amount: -25.00
ID: T003 | Date: 04/11/2026 | Category: Income | Description: Casual shift | Amount: 180.00
ID: T004 | Date: 05/11/2026 | Category: Food | De

In [10]:
test_transactions = load_transactions(WORKING_CSV)

delete_transaction(test_transactions, "T001", "viewer")

Transactions loaded successfully.
Access denied. Only Admin can delete transactions.


False

## Step 6. Evidence and personal trace
Show successful admin action, viewer denied a protected action, save/reload after fresh execution, one invalid input and Git evidence. In the report, reproduce and trace **your own delete function** by hand using the trace case assigned by the last digit of your student ID. See assessment PDF. Explain one security block using your actual variable names.

In [11]:
test_transactions = [
    t for t in load_transactions(WORKING_CSV)
    if t.transaction_id in ["T001", "T002", "T003"]
]

print("Before deletion:")
display_transactions(test_transactions)

delete_transaction(test_transactions, "T999", "admin")

print("After deletion:")
display_transactions(test_transactions)

Transactions loaded successfully.
Before deletion:

--- All Transactions ---
ID: T001 | Date: 02/11/2026 | Category: Food | Description: Groceries | Amount: -48.50
ID: T002 | Date: 03/11/2026 | Category: Transport | Description: Train card | Amount: -25.00
ID: T003 | Date: 04/11/2026 | Category: Income | Description: Casual shift | Amount: 180.00
Transaction ID not found.
After deletion:

--- All Transactions ---
ID: T001 | Date: 02/11/2026 | Category: Food | Description: Groceries | Amount: -48.50
ID: T002 | Date: 03/11/2026 | Category: Transport | Description: Train card | Amount: -25.00
ID: T003 | Date: 04/11/2026 | Category: Income | Description: Casual shift | Amount: 180.00
